In [1]:
import os
import glob
from langchain_community.document_loaders import DirectoryLoader, PDFMinerLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

C:\Users\IB200200\AppData\Local\Temp\ipykernel_21536\4152558772.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PDFMinerLoader


In [2]:
folders = glob.glob("my-knowledge/*")

documents = []
for folder in folders:
    category = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.pdf", loader_cls=PDFMinerLoader)
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["category"] = category
        doc.metadata["name"] = doc.metadata["source"].split("\\")[-1][:-4]
        documents.append(doc)

In [5]:
len(documents)

12

In [6]:
documents[5].metadata

{'producer': 'Microsoft® PowerPoint® for Microsoft 365',
 'creator': 'Microsoft® PowerPoint® for Microsoft 365',
 'creationdate': '2022-11-04T13:12:45+01:00',
 'title': 'Slide 1',
 'author': 'Ismet Maksumić',
 'moddate': '2022-11-04T13:12:45+01:00',
 'total_pages': 11,
 'source': 'my-knowledge\\razvoj_softvera\\RSI_P4_Agilne metode_SCRUM_planning.pdf',
 'category': 'razvoj_softvera',
 'name': 'RSI_P4_Agilne metode_SCRUM_planning'}

In [7]:
print(documents[5].page_content)

Razvoj softvera I
Agilne metode razvoja softvera
Planiranje sprinta

Planiranje sprinta

Backlog proizvoda

Backlog sprinta

Funkcionalnosti

Zadaci / taskovi

Prečišćavanje

Planiranje predstavlja 
prvu aktivnost 
svakog sprinta
Planiranje sprinta

• Tokom planiranja sprinta product owner kompletnom timu

prezentuje najprioritetnije funkcionalnosti product backloga

• Članovi tima postavljaju pitanja kako bi korisničke priče (koje
predstavljaju dosta generalan i kratak nivo opisa) bili u stanju
pretvoriti u zadatke backloga sprinta. Rezultat aktivnosti planiranja
jasno definisani cilj sprinta i stavke sprint
spinta trebaju biti
backloga

• Cilj sprinta predstavlja kratki opis onoga što razvojni tim treba
postići tokom sprinta, npr. : Implementirati mogućnost zakazivanja
pregleda kod ljekara uz mogućnost otkazivanja odobrenog termina,
te zahtijevanje dodjele novog termina

• Stavke sprint backloga podrazumijevaju listu konkretnih zadataka na

koje se razvojni tim obavezuje implementi

In [8]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

Created a chunk of size 1009, which is longer than the specified 1000
Created a chunk of size 1019, which is longer than the specified 1000
Created a chunk of size 1048, which is longer than the specified 1000
Created a chunk of size 1443, which is longer than the specified 1000
Created a chunk of size 1082, which is longer than the specified 1000
Created a chunk of size 1009, which is longer than the specified 1000
Created a chunk of size 1218, which is longer than the specified 1000
Created a chunk of size 1231, which is longer than the specified 1000
Created a chunk of size 1411, which is longer than the specified 1000
Created a chunk of size 1046, which is longer than the specified 1000
Created a chunk of size 1392, which is longer than the specified 1000
Created a chunk of size 1487, which is longer than the specified 1000
Created a chunk of size 1523, which is longer than the specified 1000
Created a chunk of size 1391, which is longer than the specified 1000
Created a chunk of s

In [9]:
len(chunks)

323

In [12]:
print(chunks[2].page_content)

Na primjer, pitanje arhitekture je da li će računar imati instrukciju za množenje, a organizacijsko 
pitanje je da li će ta instrukcija biti implementirana pomoću specijalne jedinice za množenje ili 
mehanizmom  višekratnog  korištenja jedinice  za sabiranje.  U  tom  slučaju,  izbor  organizacije 
može  se  zasnivati  na  očekivanoj  frekvenciji  korištenja  instrukcije  množenja,  relativnim 
brzinama  oba  pristupa,  te  cijeni  i  fizičkoj  veličini  jedinice  za  množenje.  Razlika  između 
arhitekture i organizacije je veoma važna. 

Struktura i funkcija


### Vector database creation

In [16]:
embedding_model = "jeffh/intfloat-multilingual-e5-large-instruct:f32"
db_name = "vector_db"

In [17]:
embeddings = OllamaEmbeddings(model=embedding_model)

In [ ]:
# if exist to clear

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

In [18]:
# create vectorstore

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 323 documents
